# Sentinel-2 / MSI (MultiSpectral Instrument)

Example notebook from the [Satellite Data Catalog](https://alessandrosebastianelli.github.io/satcat/satellite.html?folder=sentinel/sentinel-2-msi). Each section below runs the Python example of one processing level.

Run the cells top to bottom. Some sources need a free account; the cell that needs credentials asks for them when it runs.

## L1C

Level 1C: Top-of-atmosphere (TOA) reflectance, orthorectified and geometrically corrected — no atmospheric correction applied. Use this if you want full control over the atmospheric correction step yourself.

Level-1C top-of-atmosphere reflectance through the Earth Engine Python API (COPERNICUS/S2_HARMONIZED). Needs a Google account with Earth Engine access; the notebook asks for your Cloud project ID.

In [ ]:
%pip install -q earthengine-api

In [ ]:
import ee
from IPython.display import Image, display

# One-off sign-in with a Google account that has Earth Engine access
ee.Authenticate()
PROJECT = input("Google Cloud project ID registered for Earth Engine: ")
ee.Initialize(project=PROJECT)

point = ee.Geometry.Point([12.5, 41.9])           # Rome - change to your area
collection = (
    ee.ImageCollection("COPERNICUS/S2_HARMONIZED")
    .filterBounds(point)
    .filterDate("2024-06-01", "2024-08-31")
    .sort("CLOUDY_PIXEL_PERCENTAGE")
)
print("scenes:", collection.size().getInfo())
image = ee.Image(collection.first())
print("least cloudy scene:", image.get("system:index").getInfo(), "cloud cover:", image.get("CLOUDY_PIXEL_PERCENTAGE").getInfo())

url = image.select(['B4', 'B3', 'B2']).getThumbURL({
    "min": 0, "max": 3000, "dimensions": 900,
    "region": point.buffer(30000).bounds(),
})
display(Image(url=url))

## L2-A

Level 2A: Bottom-of-atmosphere (BOA) surface reflectance — atmospheric correction already applied by ESA's Sen2Cor processor, plus a Scene Classification (SCL) band for cloud/shadow/water masking. The level most analysis-ready workflows should start from.

Searches the Microsoft Planetary Computer STAC catalogue (no account needed), picks the least cloudy scene and plots NDVI from the surface-reflectance bands. For Sentinel-2 older than late 2021 or newer than the processing baseline change, remember that reflectance offsets differ (see notes).

In [ ]:
%pip install -q pystac-client planetary-computer rioxarray matplotlib

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import planetary_computer
import pystac_client
import rioxarray

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)


def open_overview(href, level=2):
    """Open a COG at reduced resolution (fast), falling back to full resolution."""
    try:
        return rioxarray.open_rasterio(href, overview_level=level, masked=True).squeeze()
    except Exception:
        return rioxarray.open_rasterio(href, masked=True).squeeze()

search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=[12.4, 41.8, 12.6, 42.0],                      # Rome - change to your area
    datetime="2024-06-01/2024-06-30",
    query={"eo:cloud_cover": {"lt": 20}},
)
items = search.item_collection()
if not items:
    raise SystemExit("No scenes found - widen the dates or the bounding box")
item = min(items, key=lambda i: i.properties["eo:cloud_cover"])
print(len(items), "scenes found; using", item.id, item.datetime)

red = open_overview(item.assets["B04"].href, level=2) / 10000
nir = open_overview(item.assets["B08"].href, level=2) / 10000
ndvi = (nir - red) / (nir + red)
ndvi.plot(cmap="YlGn", vmin=-0.2, vmax=0.9)
plt.title("NDVI - " + item.id)
plt.show()